# MICA benchmark: what does the map of each entry hold? (CPU survey of 20 entries)

The first pilot (entry 26DK) showed that the deposited model can be one chain while the map holds about fifteen copies of it. That changes the cost of every method and the
way the models are scored. This notebook measures it for every entry of `data/entries_2026.csv`:

1. checks the machine and the internet (no GPU, no Phenix, no secrets and no private files are needed),
2. for each entry downloads the map (EMDB) and the deposited model (RCSB) and reads the EMDB header file (contour level, resolution, symmetry) and the RCSB assembly,
3. measures the grid and box size, the volume above the contour level, how much of it lies within 3 A of the deposited model, how many copies of the model that suggests,
   the connected pieces of density, and the number of reflections that Phaser would get at the map resolution,
4. writes `survey.csv` and `survey_components.csv`, refreshes a partial zip after every entry, and packs one zip.

**What you do (Kaggle, background mode)**: accelerator **None**, Internet **On**, then *Save Version -> Save & Run All (Commit)*. It runs on Kaggle's servers; you can close
the tab and your Wi-Fi does not matter. About 30 to 60 minutes. When the version shows *Succeeded*, open it and download the zip `survey_result_....zip` from its Output.
Send it to me. **Do not stop or cancel the run** unless it is stuck for more than 3 hours.


## Step 1. Look at the machine

In [ ]:
import csv, datetime, getpass, glob, gzip, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, tarfile, threading, time, zipfile
from collections import deque

IN_KAGGLE = os.path.isdir("/kaggle/working") and not os.environ.get("MICA_WORK")      # MICA_WORK and MICA_BIG are only used by the tests
IN_COLAB = (not IN_KAGGLE) and os.path.isdir("/content") and not os.environ.get("MICA_WORK")
PLATFORM = "kaggle" if IN_KAGGLE else ("colab" if IN_COLAB else "other")
if os.environ.get("MICA_WORK"):
    WORK = OUT_DIR = os.environ["MICA_WORK"]
elif IN_KAGGLE:
    WORK, OUT_DIR = "/kaggle/working/pilot", "/kaggle/working"
elif IN_COLAB:
    WORK, OUT_DIR = "/content/pilot", "/content"
else:
    WORK = OUT_DIR = os.path.abspath("pilot")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
LOG = os.path.join(WORK, "run.log")
REQUIRE_GPU = False
NET_HOSTS = (('EBI FTP (EMDB maps and header files)', 'https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/header/emd-80558.xml', True), ('RCSB files (deposited models)', 'https://files.rcsb.org/header/26DK.cif', True), ('RCSB data API (assemblies)', 'https://data.rcsb.org/rest/v1/core/entry/26DK', False))
STATE = {"ok": True, "problem": "", "t0": time.time(), "started": datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "phenix_env": None, "phenix_version": "", "installer_sha256": "", "usalign": "", "usalign_version": "", "weights": {}, "env": {},
         "entries": {}, "gpu": ""}
SECRETS = []          # kept in memory only, to make sure they never end up in the zip


def log(msg=""):
    print(msg, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")


def fail(msg):
    STATE["ok"] = False
    STATE["problem"] = msg
    log("\n" + "!" * 72 + "\nSTOPPED: " + msg + "\nThe steps below are skipped. The last step still packs a zip with the log: send it to me.\n" + "!" * 72)


def run(cmd, env=None, every=1, cwd=None, log_file=None, timeout=None, quiet=False):
    """Runs a command, prints and logs its output as it comes. Returns (exit code, last lines, timed_out).
    env: values are set, None removes the variable. A command that runs longer than `timeout` seconds is killed with its child processes.
    Progress bars (lines with %|) are printed only now and then; quiet=True keeps the output out of the notebook (it still goes to log_file)."""
    e = dict(os.environ)
    for k, v in (env or {}).items():
        if v is None:
            e.pop(k, None)
        else:
            e[k] = v
    e["PYTHONUNBUFFERED"] = "1"
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=cwd, errors="replace",
                             start_new_session=True)
    except OSError as ex:
        log("could not start %s: %s" % (cmd[0] if isinstance(cmd, (list, tuple)) else cmd, ex))
        return 127, [str(ex)], False
    timed = {"out": False}

    def killer():
        timed["out"] = True
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except OSError:
            pass

    timer = threading.Timer(timeout, killer) if timeout else None
    if timer:
        timer.start()
    lf = open(log_file, "a", encoding="utf-8") if log_file else None
    last, n, bars = deque(maxlen=40), 0, 0
    try:
        for line in p.stdout:
            line = line.rstrip("\n")
            if len(line) > 600:
                line = line[:600] + " ..."
            last.append(line)
            n += 1
            if lf:
                lf.write(line + "\n")
                lf.flush()                                  # a program that is silent for an hour must not hide its last lines in a buffer
            if "%|" in line:
                bars += 1
                show = bars % 100 == 1
            else:
                show = every == 1 or n <= 5 or n % every == 0
            if show and not quiet:
                log(line)
        p.wait()
    finally:
        if timer:
            timer.cancel()
        if lf:
            lf.close()
        p.stdout.close()
    return p.returncode, list(last), timed["out"]


def sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, errors="replace")
        return (r.stdout + r.stderr).strip()
    except Exception as ex:
        return "ERROR: %s" % ex


def secret(name, prompt, hidden=True):
    """Colab Secrets, Kaggle Secrets, an environment variable, or a prompt, in this order. Returns "" if nothing can be found."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    try:
        return (getpass.getpass(prompt) if hidden else input(prompt)).strip()
    except Exception:
        return ""


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def meminfo_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return round(int(line.split()[1]) / 1048576, 1)
    except OSError:
        pass
    return None


def pick_big_dir():
    """The folder for everything large (Phenix, Python environment, MICA): the disk with the most free space."""
    if os.environ.get("MICA_BIG"):
        return os.environ["MICA_BIG"]
    best = None
    for base in ("/kaggle/temp", "/tmp", "/content", WORK):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            free = shutil.disk_usage(base).free
            if best is None or free > best[1] + 5e9:
                best = (base, free)
    return os.path.join(best[0], "mica_pilot")


BIG = pick_big_dir()
os.makedirs(BIG, exist_ok=True)


class Monitor(threading.Thread):
    """Samples GPU memory, GPU load and free RAM every few seconds into a csv file; summary() gives the peaks."""

    def __init__(self, path, every=5):
        super().__init__(daemon=True)
        self.path, self.every, self.stop_flag = path, every, threading.Event()
        self.peak_gpu_mb, self.peak_util, self.min_ram_gb, self.samples = 0, 0, None, 0

    def sample(self):
        gpu_mb, util = "", ""
        if shutil.which("nvidia-smi"):
            out = sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits", timeout=20).splitlines()
            try:
                a, b = [x.strip() for x in out[0].split(",")]
                gpu_mb, util = int(a), int(b)
                self.peak_gpu_mb, self.peak_util = max(self.peak_gpu_mb, gpu_mb), max(self.peak_util, util)
            except (ValueError, IndexError):
                gpu_mb, util = "", ""
        ram = ""
        try:
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemAvailable"):
                        ram = round(int(line.split()[1]) / 1048576, 2)
        except OSError:
            pass
        if ram != "":
            self.min_ram_gb = ram if self.min_ram_gb is None else min(self.min_ram_gb, ram)
        self.samples += 1
        with open(self.path, "a", encoding="utf-8") as f:
            f.write("%s,%s,%s,%s\n" % (datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M:%S"), gpu_mb, util, ram))

    def run(self):
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, "w", encoding="utf-8") as f:
            f.write("time_utc,gpu_mem_used_mb,gpu_util_pct,ram_available_gb\n")
        while not self.stop_flag.is_set():
            try:
                self.sample()
            except Exception:
                pass
            self.stop_flag.wait(self.every)

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)
        return {"peak_gpu_mb": self.peak_gpu_mb, "peak_gpu_util_pct": self.peak_util, "min_ram_available_gb": self.min_ram_gb, "samples": self.samples}


def check_internet():
    import urllib.request
    bad_required = []
    for name, url, required in NET_HOSTS:
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "mica-benchmark-pilot"}), timeout=20) as r:
                r.read(512)
                log("  reachable: %-9s %s" % (name, r.status))
        except Exception as ex:
            log("  NOT reachable: %-9s %s" % (name, str(ex)[:100]))
            if required:
                bad_required.append(name)
    return bad_required


free_big, free_work = round(shutil.disk_usage(BIG).free / 1e9), round(shutil.disk_usage(WORK).free / 1e9)
cpu_model = sh("lscpu | grep -m1 'Model name'").split(":")[-1].strip()
log("platform %s | python %s | %s | %s CPU cores (%s) | %s GB RAM" % (PLATFORM, platform.python_version(), platform.platform(), os.cpu_count(), cpu_model, meminfo_gb()))
log("large files go to %s (%d GB free); results to %s (%d GB free)" % (BIG, free_big, WORK, free_work))
with open("machine.txt", "w", encoding="utf-8") as f:
    f.write(sh("nvidia-smi") + "\n\n" + sh("lscpu") + "\n\n" + sh("free -h") + "\n\n" + sh("df -h") + "\n\n" + sh("g++ --version | head -1; git --version; uname -a; cat /etc/os-release | head -3") + "\n")
STATE["gpu"] = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") if shutil.which("nvidia-smi") else ""
if STATE["gpu"] and not STATE["gpu"].startswith("ERROR"):
    log("GPU: " + STATE["gpu"].replace("\n", " | "))
elif REQUIRE_GPU:
    fail("this session has no GPU. Kaggle: open Session options on the right, set Accelerator to GPU T4 x2 (or GPU P100), then Run all again. "
         "Colab: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
if STATE["ok"]:
    log("checking the internet:")
    missing = check_internet() if NET_HOSTS else []
    if "PyPI" in missing or "GitHub" in missing:
        fail("the internet is off. Kaggle: open Session options on the right and switch Internet on (the account must be phone-verified), then Run all again.")
    elif missing:
        log("WARNING: %s cannot be reached from here; the step that needs it will tell you." % ", ".join(missing))


## Step 2. Write the scripts and the frozen data files

The files below are exact copies of the files in the repository (scripts, the AlphaFold Server job table and the hashes of the 26 predictions,
the entry list, the exact list of Python packages). The last cell of this step checks their hashes.

In [ ]:
import os
for d in ("scripts", "data/af3", "env", "logs", "gpu"):
    os.makedirs(d, exist_ok=True)
print("folders ready")


In [ ]:
%%writefile scripts/nbtools.py
#!/usr/bin/env python3
"""Tools that the Phase 4 notebooks share. They were added after the first MICA pilot, which showed three weaknesses:

1. a program that is silent for an hour (Phenix docking) looked like a frozen notebook: `Heartbeat` prints one line every few minutes
   (elapsed time, load, free RAM, the busiest processes, the last line of the program's own log);
2. the zip with the results was only written by the very last cell, so a stopped session left nothing: `pack` writes the zip in
   one step and the notebooks call it after every stage (a fixed name, replaced each time);
3. the last cell used variables of earlier cells and failed after the kernel was restarted: `recover` needs nothing from the notebook,
   it finds the work folder on disk and packs whatever is there.

Everything the zip should hold is copied into the work folder by the notebook, so that these functions only walk that folder.
Not packed: maps and other large data, files over 60 MB, the AlphaFold Server predictions of the user (they must stay private), and any file
that contains one of the secrets (the Phenix password).

    python scripts/nbtools.py --work <work folder> --out <folder for the zip> --notebook <name>      # the recovery path, by hand
"""
from __future__ import annotations

import argparse
import datetime
import json
import os
import subprocess
import sys
import threading
import time
import zipfile

MAX_FILE = 60 * 1000 * 1000
SKIP_DIRS = {"AF3_results", "af3_upload", "af3_unzipped", "grids", "AF3_encodings", "temp_maps", "native", "downloads", "scripts", "data", "env", "__pycache__"}
SKIP_SUFFIXES = (".map", ".mrc", ".ccp4", ".gz", ".part", ".tmp", ".zip")
WORK_CANDIDATES = ("/kaggle/working/pilot", "/content/pilot")


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)


def last_line(path, max_chars=160):
    """The last non-empty line of a text file, or "" if there is none or the file cannot be read."""
    try:
        with open(path, "rb") as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - 8192))
            tail = f.read().decode("utf-8", errors="replace")
    except OSError:
        return ""
    lines = [ln.strip() for ln in tail.splitlines() if ln.strip()]
    return lines[-1][:max_chars] if lines else ""


def machine_load():
    """(load average over 1 minute, free RAM in GB, text about the three busiest processes), each None or "" where it cannot be read."""
    load = free = None
    try:
        with open("/proc/loadavg") as f:
            load = float(f.read().split()[0])
    except (OSError, ValueError, IndexError):
        pass
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    free = round(int(line.split()[1]) / 1048576, 1)
    except (OSError, ValueError, IndexError):
        pass
    top = ""
    try:
        r = subprocess.run(["ps", "-eo", "pcpu,etimes,comm", "--sort=-pcpu"], capture_output=True, text=True, timeout=10, errors="replace")
        rows = [ln.split(None, 2) for ln in r.stdout.splitlines()[1:4] if ln.strip()]
        top = ", ".join("%s %s%% (%ss)" % (c[2][:18], c[0], c[1]) for c in rows if len(c) == 3)
    except (OSError, subprocess.SubprocessError):
        pass
    return load, free, top


class Heartbeat(threading.Thread):
    """While a long program runs in the main thread, prints one line every `every` seconds through `log`. `watch` is a list of files whose last
    line is shown (the program's own log); `on_tick` is called after each line (the notebooks use it to refresh the partial zip)."""

    def __init__(self, label, log, watch=(), every=120, on_tick=None):
        super().__init__(daemon=True)
        self.label, self.log, self.watch, self.every, self.on_tick = label, log, list(watch), every, on_tick
        self.stop_flag, self.t0, self.ticks = threading.Event(), time.time(), 0

    def line(self):
        load, free, top = machine_load()
        minutes = (time.time() - self.t0) / 60
        parts = ["[%s] running for %.0f min" % (self.label, minutes) if minutes >= 1 else "[%s] running for %.0f s" % (self.label, time.time() - self.t0)]
        if load is not None:
            parts.append("load %.1f" % load)
        if free is not None:
            parts.append("free RAM %.1f GB" % free)
        if top:
            parts.append("busiest: " + top)
        for path in self.watch:
            if os.path.exists(path):
                age = time.time() - os.path.getmtime(path)
                parts.append("%s: %d bytes, changed %s ago, last line: %s" % (os.path.basename(path), os.path.getsize(path),
                                                                           "%.0f s" % age if age < 120 else "%.0f min" % (age / 60), last_line(path) or "(empty)"))
        return " | ".join(parts)

    def run(self):
        while not self.stop_flag.wait(self.every):
            try:
                self.log(self.line())
                self.ticks += 1
                if self.on_tick:
                    self.on_tick()
            except Exception as ex:                       # a failing heartbeat must never stop the program it watches
                try:
                    self.log("[%s] heartbeat problem: %s" % (self.label, str(ex)[:150]))
                except Exception:
                    pass

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)


def collect(work, skip_dirs=SKIP_DIRS, skip_suffixes=SKIP_SUFFIXES):
    """All files below `work` that may go into the zip, in a fixed order."""
    files = []
    for dirpath, dirs, names in os.walk(work):
        dirs[:] = sorted(d for d in dirs if d not in skip_dirs)
        for n in sorted(names):
            if not n.endswith(skip_suffixes):
                files.append(os.path.join(dirpath, n))
    return files


def pack(work, zip_path, root_name, secrets=(), max_file=MAX_FILE, extra=()):
    """Writes `zip_path` in one step (a temporary file that replaces the old zip at the end, so a stop in the middle never leaves a broken zip).
    Returns (number of files, list of what was left out and why). `extra` are files outside `work` that go in as well."""
    wanted = collect(work) + [p for p in extra if os.path.isfile(p)]
    left_out, tmp = [], zip_path + ".tmp"
    try:
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for path in dict.fromkeys(wanted):
                try:
                    if os.path.getsize(path) > max_file:
                        left_out.append("%s (larger than %d MB)" % (os.path.relpath(path, work), max_file // 1000000))
                        continue
                    with open(path, "rb") as f:
                        data = f.read()
                except OSError as ex:
                    left_out.append("%s (%s)" % (os.path.relpath(path, work), str(ex)[:60]))
                    continue
                if any(s and s.encode() in data for s in secrets):
                    left_out.append("%s (contained the password)" % os.path.relpath(path, work))
                    continue
                inside = os.path.relpath(path, work) if path.startswith(work + os.sep) else os.path.join("outside", path.lstrip(os.sep))
                z.writestr(root_name + "/" + inside, data)
            n = len(z.namelist())
        os.replace(tmp, zip_path)
    except BaseException:
        try:
            os.remove(tmp)
        except OSError:
            pass
        raise
    return n, left_out


def partial_zip_path(out_dir, notebook):
    return os.path.join(out_dir, "%s_partial.zip" % notebook)


def final_pack(work, out_dir, notebook, complete, secrets=(), extra=(), note=""):
    """The zip that is sent to me. complete: True (everything ran), False (something stopped) or None (unknown: recovered after a restart).
    The partial zip is removed once this one exists."""
    stamp = utc_now().strftime("%Y%m%d-%H%M")
    suffix = "" if complete else ("_INCOMPLETE" if complete is False else "_RECOVERED")
    name = "%s_result_%s%s" % (notebook, stamp, suffix)
    path = os.path.join(out_dir, name + ".zip")
    n, left_out = pack(work, path, name, secrets=secrets, extra=extra)
    try:
        os.remove(partial_zip_path(out_dir, notebook))
    except OSError:
        pass
    return path, n, left_out


def find_work(env=None):
    """The work folder of a notebook that ran in this session: the one with a run.log."""
    env = os.environ if env is None else env
    for cand in (env.get("MICA_WORK"),) + WORK_CANDIDATES + (os.path.abspath("pilot"),):
        if cand and os.path.isfile(os.path.join(cand, "run.log")):
            return cand
    return None


def known_secrets(env=None):
    """The Phenix password, if it can be read again (environment, Kaggle or Colab secrets), to check the files for it after a restart."""
    env = os.environ if env is None else env
    found = []
    if env.get("PHENIX_PASSWORD"):
        found.append(env["PHENIX_PASSWORD"].strip())
    for getter in ("kaggle", "colab"):
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                v = UserSecretsClient().get_secret("PHENIX_PASSWORD")
            else:
                from google.colab import userdata
                v = userdata.get("PHENIX_PASSWORD")
            if v:
                found.append(v.strip())
        except Exception:
            pass
    return [s for s in dict.fromkeys(found) if len(s) >= 6]


def read_state(work):
    try:
        with open(os.path.join(work, "state.json"), encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def recover(notebook, out_dir=None, env=None, say=print):
    """The last step when nothing from the notebook is left in memory (kernel restarted, cell run alone, ...). Returns the zip path or None."""
    env = os.environ if env is None else env
    work = find_work(env)
    if not work:
        say("RECOVERY: no work folder with a run.log was found (looked in %s). Nothing to pack." % ", ".join(WORK_CANDIDATES))
        return None
    out_dir = out_dir or env.get("MICA_WORK") or os.path.dirname(work)
    state = read_state(work)
    info = {"notebook": notebook, "recovered": True, "recovered_utc": utc_now().isoformat(), "state_json_found": bool(state), "state": state}
    with open(os.path.join(work, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    path, n, left_out = final_pack(work, out_dir, notebook, complete=None, secrets=known_secrets(env))
    say("RECOVERY: zip written from the files that are on disk: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        say("left out of the zip: %s" % "; ".join(left_out))
    say("If a notebook cell stopped with an error, send me this zip and the text of the error.")
    return path


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--work", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--notebook", required=True)
    args = p.parse_args(argv)
    path, n, left_out = final_pack(args.work, args.out, args.notebook, complete=None)
    print("%s (%d files)" % (path, n))
    for x in left_out:
        print("left out:", x)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/map_survey.py
#!/usr/bin/env python3
"""What does a map hold, compared with the deposited model? (numpy and scipy only; no mrcfile, no Biopython, no Phenix)

Why it exists. In the first pilot, entry 26DK (one deposited chain of 191 residues) turned out to sit in a map that holds about fifteen copies of it, because the
deposited model is the asymmetric unit of a symmetric assembly. Every method then sees much more density than the one chain that the PDB model describes, the
Phenix docking of MICA works on a 265 A cube, and a method that builds every copy is scored against one chain. Before the full runs we need to know, for every
entry: how big the map is, how many copies of the deposited model it holds, and how many reflections Phaser would have to handle.

What it does.
  read_mrc(path)            the map as float32 [z][y][x] with grid, voxel size, start indices and origin (CCP4/MRC2014 header, any axis order, any byte order)
  read_atoms(cif)           protein atoms of the first model of an mmCIF file (an own small parser, standard residues only)
  choose_frame(...)         the offset of the map in the coordinates of the model: start indices, ORIGIN field or both, whichever puts the C-alpha atoms into
                            the highest density (a map and a model of one entry always agree in one of the three)
  survey(map, cif, ...)     the numbers per entry: volume above the contour level, the part of it within `radius` A of a model atom, the copies of the model that
                            the volume suggests, connected components, extent of the content, reflections that Phaser would get at the map resolution

    python scripts/map_survey.py <map or map.gz> <model.cif> [--contour X] [--resolution Y]       # one entry, prints the numbers
"""
from __future__ import annotations

import argparse
import gzip
import json
import math
import os
import shlex
import shutil
import struct
import sys
import time

import numpy as np

MODE_DTYPES = {0: "i1", 1: "i2", 2: "f4", 6: "u2", 12: "f2"}
AMINO_ACIDS = {"ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL", "MSE", "UNK",
               "SEC", "PYL"}
REFERENCE_REFLECTIONS = 915139          # Phaser's count for entry 26DK in the first docking test (265.6 A cube, 3.5 A)

SURVEY_COLUMNS = [
    "entry", "emdb_id", "resolution_table_A", "resolution_header_A", "contour_level", "contour_source", "grid_x", "grid_y", "grid_z", "voxel_A", "box_x_A", "box_y_A", "box_z_A",
    "map_mode", "map_min", "map_max", "map_mean", "map_std", "map_file_mb", "frame", "frame_scores", "ca_in_map_frac", "ca_mean_density_over_contour",
    "n_chains", "n_ca", "n_heavy_atoms", "model_extent_A", "above_volume_A3", "above_in_model_A3", "above_fraction_in_model", "copies_of_model_estimate",
    "components_ge_500A3", "components_ge_5000A3", "components_touching_model", "largest_component_A3", "largest_component_fraction", "content_x_A", "content_y_A",
    "content_z_A", "content_fraction_of_box", "reflections_full_M", "reflections_content_M", "docking_load_vs_26DK", "point_group", "number_of_copies_header",
    "oligomeric_state_header", "sample_name", "assembly_count", "assembly1_polymer_instances", "assembly1_oligomeric_count", "symmetry_rcsb", "map_sha256", "seconds", "note"]


# --------------------------------------------------------------------------------------------------------------------------------- the map
def read_mrc(path):
    """(volume, info). volume: float32 array indexed [z][y][x] (crystal axes), whatever the axis order of the file. info: grid_zyx, voxel (x, y, z in A),
    nstart (x, y, z, in voxels), origin (x, y, z, in A), cell (A), mode, axes (mapc, mapr, maps), little_endian, data_offset."""
    with open(path, "rb") as f:
        head = f.read(1024)
    if len(head) < 1024:
        raise ValueError("%s is shorter than an MRC header (%d bytes)" % (path, len(head)))
    parsed = None
    for little in (head[212] == 0x44, head[212] != 0x44):      # the machine stamp (0x44 little, 0x11 big endian) first, the other order as a second guess
        e = "<" if little else ">"
        nx, ny, nz, mode, nxs, nys, nzs, mx, my, mz = struct.unpack(e + "10i", head[0:40])
        if mode in MODE_DTYPES and 0 < nx < 20000 and 0 < ny < 20000 and 0 < nz < 20000:
            parsed = (e, nx, ny, nz, mode, nxs, nys, nzs, mx, my, mz)
            break
    if parsed is None:
        raise ValueError("%s: the header does not look like an MRC/CCP4 map (mode %s, grid %s x %s x %s)" % (path, mode, nx, ny, nz))
    e, nx, ny, nz, mode, nxs, nys, nzs, mx, my, mz = parsed
    cell = struct.unpack(e + "3f", head[40:52])
    mapc, mapr, maps = struct.unpack(e + "3i", head[64:76])
    if sorted((mapc, mapr, maps)) != [1, 2, 3]:
        mapc, mapr, maps = 1, 2, 3                                              # a header without axis order: the standard one
    nsymbt = struct.unpack(e + "i", head[92:96])[0]
    origin = struct.unpack(e + "3f", head[196:208])
    offset = 1024 + max(nsymbt, 0)
    raw = np.memmap(path, dtype=np.dtype(e + MODE_DTYPES[mode]), mode="r", offset=offset, shape=(nz, ny, nx))      # sections, rows, columns
    crystal_of_axis = [maps, mapr, mapc]
    perm = [crystal_of_axis.index(3), crystal_of_axis.index(2), crystal_of_axis.index(1)]
    vol = np.array(np.transpose(raw, perm), dtype=np.float32)
    del raw
    n_by_crystal = {mapc: nx, mapr: ny, maps: nz}
    m_by_crystal = {1: mx, 2: my, 3: mz}                                         # MX, MY, MZ and the cell edges are by crystal axis, NX, NY, NZ and the starts by storage order
    start_by_crystal = {mapc: nxs, mapr: nys, maps: nzs}
    voxel = []
    for k in (1, 2, 3):
        grid = m_by_crystal[k] if m_by_crystal[k] > 0 else n_by_crystal[k]
        voxel.append(cell[k - 1] / grid if cell[k - 1] > 0 else 1.0)
    info = {"grid_zyx": tuple(int(v) for v in vol.shape), "voxel": tuple(float(v) for v in voxel), "nstart": tuple(int(start_by_crystal[k]) for k in (1, 2, 3)),
            "origin": tuple(float(v) for v in origin), "cell": tuple(float(v) for v in cell), "mode": int(mode), "axes": (mapc, mapr, maps), "little_endian": e == "<",
            "data_offset": offset}
    return vol, info


def gunzip_if_needed(path, dest):
    """`path` itself when it is a plain file; otherwise the unzipped copy `dest` (made once)."""
    with open(path, "rb") as f:
        magic = f.read(2)
    if magic != b"\x1f\x8b":
        return path
    if not (os.path.exists(dest) and os.path.getsize(dest) > 1024):
        os.makedirs(os.path.dirname(os.path.abspath(dest)), exist_ok=True)
        with gzip.open(path, "rb") as src, open(dest + ".part", "wb") as out:
            shutil.copyfileobj(src, out, 1 << 22)
        os.replace(dest + ".part", dest)
    return dest


# ------------------------------------------------------------------------------------------------------------------------------- the model
def read_atoms(cif_path):
    """Protein atoms of the first model of an mmCIF file: {"ca": (n, 3) array, "heavy": (m, 3) array, "chains": {chain id: residues with a C-alpha atom}}.
    Alternative locations other than the first are skipped, hydrogens too. Only the 20 standard amino acids (and MSE, SEC, PYL, UNK) count."""
    cols, ca, heavy, chains = [], [], [], {}
    state, first_model = 0, None
    with open(cif_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            if state == 0:
                if line.startswith("loop_"):
                    cols, state = [], 1
                continue
            if state == 1:                                                            # the column names of a loop
                if line.startswith("_atom_site."):
                    cols.append(line.split()[0][len("_atom_site."):])
                    continue
                if line.startswith("_") or line.startswith("loop_") or not cols:
                    cols, state = [], (1 if line.startswith("loop_") else 0)
                    continue
                state = 2                                                             # first data line of an atom_site loop
            if state == 2:
                if not line.strip() or line.startswith(("#", "loop_", "_", "data_")):
                    break
                tok = line.split()
                if len(tok) != len(cols):
                    try:
                        tok = shlex.split(line)
                    except ValueError:
                        continue
                    if len(tok) != len(cols):
                        continue
                row = dict(zip(cols, tok))
                if row.get("group_PDB", "ATOM") not in ("ATOM", "HETATM"):
                    continue
                model = row.get("pdbx_PDB_model_num", "1")
                if first_model is None:
                    first_model = model
                elif model != first_model:
                    break
                if row.get("label_comp_id", "").upper() not in AMINO_ACIDS:                  # water, ligands, nucleic acids (MSE is a HETATM amino acid and stays)
                    continue
                if row.get("label_alt_id", ".") not in (".", "?", "A", ""):
                    continue
                if row.get("type_symbol", "C").upper() in ("H", "D"):
                    continue
                try:
                    xyz = (float(row["Cartn_x"]), float(row["Cartn_y"]), float(row["Cartn_z"]))
                except (KeyError, ValueError):
                    continue
                heavy.append(xyz)
                if row.get("label_atom_id", "").strip('"') == "CA":
                    ca.append(xyz)
                    chain = row.get("auth_asym_id", row.get("label_asym_id", "?"))
                    chains[chain] = chains.get(chain, 0) + 1
    if not cols or not heavy:
        raise ValueError("%s: no protein atoms found in an _atom_site loop" % cif_path)
    return {"ca": np.array(ca, dtype=np.float64).reshape(-1, 3), "heavy": np.array(heavy, dtype=np.float64).reshape(-1, 3), "chains": chains}


# ------------------------------------------------------------------------------------------------------------------------------ the frame
def frame_candidates(info):
    """The offsets (A) of the first voxel in model coordinates that a header can mean: the start indices, the ORIGIN field, or both."""
    voxel = np.array(info["voxel"], dtype=float)
    start = np.array(info["nstart"], dtype=float) * voxel
    origin = np.array(info["origin"], dtype=float)
    cands = {"start": start, "origin": origin, "start+origin": start + origin}
    out, seen = {}, []
    for name, v in cands.items():
        if not any(np.allclose(v, s) for s in seen):
            seen.append(v)
            out[name] = v
    return out


def density_at(vol, info, offset, xyz):
    """Trilinear density of the map at the points `xyz` (n, 3) in model coordinates; 0 outside the map. Also the fraction of the points inside."""
    from scipy.ndimage import map_coordinates
    voxel = np.array(info["voxel"], dtype=float)
    idx = (np.asarray(xyz, dtype=float) - np.asarray(offset, dtype=float)) / voxel           # (ix, iy, iz)
    nz, ny, nx = vol.shape
    inside = (idx[:, 0] >= 0) & (idx[:, 0] <= nx - 1) & (idx[:, 1] >= 0) & (idx[:, 1] <= ny - 1) & (idx[:, 2] >= 0) & (idx[:, 2] <= nz - 1)
    values = map_coordinates(vol, [idx[:, 2], idx[:, 1], idx[:, 0]], order=1, mode="constant", cval=0.0)
    return values, float(inside.mean()) if len(idx) else 0.0


def choose_frame(vol, info, ca_xyz):
    """(name, offset, scores) where scores = {name: mean density at the C-alpha atoms}; the best one wins, the first listed on a tie."""
    scores, best = {}, None
    for name, off in frame_candidates(info).items():
        values, _ = density_at(vol, info, off, ca_xyz)
        scores[name] = float(values.mean()) if len(values) else float("nan")
        if best is None or scores[name] > scores[best[0]] + 1e-9:
            best = (name, off)
    return best[0], best[1], scores


def model_mask(shape_zyx, info, offset, xyz, radius):
    """Boolean array [z][y][x]: voxels whose centre is within `radius` A of one of the points `xyz`."""
    voxel = np.array(info["voxel"], dtype=float)
    mask = np.zeros(shape_zyx, dtype=bool)
    nz, ny, nx = shape_zyx
    idx = (np.asarray(xyz, dtype=float) - np.asarray(offset, dtype=float)) / voxel
    reach = np.ceil(radius / voxel).astype(int) + 1
    r2 = radius * radius
    for c in idx:
        lo = np.maximum(np.floor(c - reach).astype(int), 0)
        hi = np.minimum(np.ceil(c + reach).astype(int) + 1, [nx, ny, nz])
        if np.any(hi <= lo):
            continue
        gx = (np.arange(lo[0], hi[0]) - c[0]) * voxel[0]
        gy = (np.arange(lo[1], hi[1]) - c[1]) * voxel[1]
        gz = (np.arange(lo[2], hi[2]) - c[2]) * voxel[2]
        d2 = gz[:, None, None] ** 2 + gy[None, :, None] ** 2 + gx[None, None, :] ** 2
        mask[lo[2]:hi[2], lo[1]:hi[1], lo[0]:hi[0]] |= d2 <= r2
    return mask


def reflections(edges_A, d):
    """Number of unique reflections of a P1 cell with these edges (A) up to resolution d (A), Friedel pairs counted once: V * 4 pi / 3 / d^3 / 2.
    Phaser reported 915,139 for the 265.6 A cube of entry 26DK at 3.5 A; this formula gives 915,000."""
    volume = float(edges_A[0]) * float(edges_A[1]) * float(edges_A[2])
    return volume * (4.0 * math.pi / 3.0) / (d ** 3) / 2.0


# ---------------------------------------------------------------------------------------------------------------------------- the survey
def survey(map_path, cif_path, contour=None, resolution=None, radius=3.0, min_small_A3=500.0, min_large_A3=5000.0, top_components=15):
    """The numbers of one entry (a dict with the keys of SURVEY_COLUMNS that this function knows, plus "components": a list of the largest components)."""
    t0 = time.time()
    vol, info = read_mrc(map_path)
    atoms = read_atoms(cif_path)
    voxel = np.array(info["voxel"], dtype=float)
    voxel_volume = float(np.prod(voxel))
    nz, ny, nx = vol.shape
    res = {"grid_x": nx, "grid_y": ny, "grid_z": nz, "voxel_A": round(float(np.mean(voxel)), 4), "map_mode": info["mode"],
           "box_x_A": round(nx * voxel[0], 1), "box_y_A": round(ny * voxel[1], 1), "box_z_A": round(nz * voxel[2], 1),
           "map_min": float(vol.min()), "map_max": float(vol.max()), "map_mean": float(vol.mean()), "map_std": float(vol.std())}
    note = []
    name, offset, scores = choose_frame(vol, info, atoms["ca"])
    ca_density, inside = density_at(vol, info, offset, atoms["ca"])
    res.update(frame=name, frame_scores=json.dumps({k: round(v, 5) for k, v in scores.items()}), ca_in_map_frac=round(inside, 4))
    if contour is None:
        contour = res["map_mean"] + 3.0 * res["map_std"]
        res["contour_source"] = "mean + 3 std (no level in the header file)"
        note.append("contour level taken as mean + 3 std")
    else:
        res["contour_source"] = "EMDB header file"
    res["contour_level"] = float(contour)
    res["ca_mean_density_over_contour"] = round(float(ca_density.mean()) / contour, 3) if contour else ""
    if inside < 0.9:
        note.append("only %.0f%% of the C-alpha atoms lie inside the map in the frame '%s': the model and the map may not be in one frame" % (100 * inside, name))
    if contour and float(ca_density.mean()) < 0.5 * contour:
        note.append("the C-alpha atoms sit at low density (mean %.3g, contour %.3g): check the frame or the contour level" % (float(ca_density.mean()), contour))
    above = vol > contour
    n_above = int(above.sum())
    mask = model_mask(vol.shape, info, offset, atoms["heavy"], radius)
    in_model = int((above & mask).sum())
    res.update(n_chains=len(atoms["chains"]), n_ca=int(len(atoms["ca"])), n_heavy_atoms=int(len(atoms["heavy"])),
               model_extent_A=round(float(np.linalg.norm(atoms["ca"].max(axis=0) - atoms["ca"].min(axis=0))), 1),
               above_volume_A3=round(n_above * voxel_volume), above_in_model_A3=round(in_model * voxel_volume))
    res["above_fraction_in_model"] = round(in_model / n_above, 4) if n_above else ""
    res["copies_of_model_estimate"] = round(n_above / in_model, 2) if in_model else ""
    comps = []
    if n_above:
        from scipy.ndimage import find_objects, label
        lab, n = label(above)
        sizes = np.bincount(lab.ravel(), minlength=n + 1)[1:]
        touching = np.bincount(lab[mask & above], minlength=n + 1)[1:]
        order = np.argsort(-sizes)
        big_small = sizes * voxel_volume >= min_small_A3
        big_large = sizes * voxel_volume >= min_large_A3
        res.update(components_ge_500A3=int(big_small.sum()), components_ge_5000A3=int(big_large.sum()),
                   components_touching_model=int(((touching > 0) & big_small).sum()), largest_component_A3=round(float(sizes[order[0]]) * voxel_volume),
                   largest_component_fraction=round(float(sizes[order[0]]) / n_above, 4))
        objs = find_objects(lab, max_label=n)
        for rank, k in enumerate(order[:top_components], 1):
            sl = objs[k]
            sub = lab[sl] == (k + 1)
            zz, yy, xx = np.nonzero(sub)
            centre = np.array([xx.mean() + sl[2].start, yy.mean() + sl[1].start, zz.mean() + sl[0].start]) * voxel + offset
            comps.append({"rank": rank, "volume_A3": round(float(sizes[k]) * voxel_volume), "voxels_in_model_region": int(touching[k]),
                          "centre_x": round(float(centre[0]), 1), "centre_y": round(float(centre[1]), 1), "centre_z": round(float(centre[2]), 1)})
        del lab
        zs, ys, xs = np.nonzero(above.any(axis=(1, 2))), np.nonzero(above.any(axis=(0, 2))), np.nonzero(above.any(axis=(0, 1)))
        ext = [(a[0].max() - a[0].min() + 1) * v for a, v in ((xs, voxel[0]), (ys, voxel[1]), (zs, voxel[2]))]
        res.update(content_x_A=round(float(ext[0]), 1), content_y_A=round(float(ext[1]), 1), content_z_A=round(float(ext[2]), 1),
                   content_fraction_of_box=round(float(np.prod(ext)) / float(nx * ny * nz * voxel_volume), 4))
        if resolution:
            res["reflections_content_M"] = round(reflections(ext, resolution) / 1e6, 3)
    if resolution:
        full = reflections((nx * voxel[0], ny * voxel[1], nz * voxel[2]), resolution)
        res.update(reflections_full_M=round(full / 1e6, 3), docking_load_vs_26DK=round(full / REFERENCE_REFLECTIONS, 2))
    res.update(seconds=round(time.time() - t0, 1), note="; ".join(note), components=comps)
    return res


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("map")
    p.add_argument("cif")
    p.add_argument("--contour", type=float)
    p.add_argument("--resolution", type=float)
    p.add_argument("--radius", type=float, default=3.0)
    args = p.parse_args(argv)
    path = gunzip_if_needed(args.map, os.path.splitext(os.path.abspath(args.map))[0])
    res = survey(path, args.cif, args.contour, args.resolution, args.radius)
    comps = res.pop("components")
    print(json.dumps(res, indent=1))
    for c in comps:
        print(c)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile data/entries_2026.csv
emdb_id,pdb_id,emdb_release_date,pdb_release_date,resolution_A,stratum,modeled_residues,unique_chain_seqs,unk_fraction,cc_mask,cluster_id,seed_rank,max_id_to_pre2026_pdb,cluster_size,longest_seq_len,n_chains,map_size_mb,cif_url,map_url
EMD-55586,9T5K,2026-04-22,2026-04-22,2.6,le3.0,381,1,0.0,0.8200,9T5K,1,0.614,1,453,1,250.0,https://files.rcsb.org/download/9T5K.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55586/map/emd_55586.map.gz
EMD-68251,22FX,2026-03-04,2026-03-04,1.24,le3.0,173,1,0.0,0.7388,22FX,2,1.000,1,182,1,75.2,https://files.rcsb.org/download/22FX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68251/map/emd_68251.map.gz
EMD-61564,9JKK,2026-04-08,2026-04-08,3.0,le3.0,530,1,0.0,0.8420,9W97,3,1.000,39,620,1,65.5,https://files.rcsb.org/download/9JKK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61564/map/emd_61564.map.gz
EMD-56022,9TKA,2026-07-29,2026-07-29,2.7,le3.0,247,1,0.0,0.7625,9THK,4,0.000,3,293,1,182.3,https://files.rcsb.org/download/9TKA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56022/map/emd_56022.map.gz
EMD-65930,9WFL,2026-04-01,2026-04-01,2.75,le3.0,682,2,0.0,0.8396,9WF3,5,1.000,6,697,2,65.5,https://files.rcsb.org/download/9WFL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65930/map/emd_65930.map.gz
EMD-65077,9VI4,2026-03-11,2026-03-11,1.8,le3.0,516,1,0.0,0.8749,9P4L,6,0.984,9,516,1,1024.0,https://files.rcsb.org/download/9VI4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65077/map/emd_65077.map.gz
EMD-74403,9ZLH,2026-09-16,2026-09-16,2.14,le3.0,206,1,0.0,0.8468,9ZK8,7,1.000,7,206,1,65.5,https://files.rcsb.org/download/9ZLH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74403/map/emd_74403.map.gz
EMD-65363,9VUH,2026-07-15,2026-07-15,2.7,le3.0,634,3,0.0,0.7439,9VUH,8,1.000,1,365,3,128.0,https://files.rcsb.org/download/9VUH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65363/map/emd_65363.map.gz
EMD-70305,9OC0,2026-05-06,2026-05-06,2.97,le3.0,512,2,0.0,0.8262,9OBT,11,1.000,8,862,4,332.8,https://files.rcsb.org/download/9OC0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70305/map/emd_70305.map.gz
EMD-76936,13BD,2026-06-17,2026-06-17,2.65,le3.0,461,1,0.0,0.8841,9OVK,12,0.954,25,1271,2,281.2,https://files.rcsb.org/download/13BD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76936/map/emd_76936.map.gz
EMD-66160,9WQ6,2026-09-16,2026-09-16,3.4,gt3.0_le4.0,261,1,0.0,0.7629,9W3F,4,1.000,11,339,1,128.0,https://files.rcsb.org/download/9WQ6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66160/map/emd_66160.map.gz
EMD-64777,9V4D,2026-07-01,2026-07-01,3.07,gt3.0_le4.0,276,2,0.0,0.8202,25IF,5,0.503,6,393,2,54.0,https://files.rcsb.org/download/9V4D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64777/map/emd_64777.map.gz
EMD-70828,9OTA,2026-05-27,2026-05-27,3.2,gt3.0_le4.0,534,1,0.0,0.8452,9E94,7,1.000,2,270,2,68.7,https://files.rcsb.org/download/9OTA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70828/map/emd_70828.map.gz
EMD-62507,9KQM,2026-06-03,2026-06-03,3.5,gt3.0_le4.0,403,1,0.0,0.7874,9KKN,8,0.997,9,637,1,65.5,https://files.rcsb.org/download/9KQM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62507/map/emd_62507.map.gz
EMD-66088,9WME,2026-09-23,2026-09-23,3.2,gt3.0_le4.0,446,2,0.0,0.7050,9WME,9,1.000,1,657,2,22.8,https://files.rcsb.org/download/9WME.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66088/map/emd_66088.map.gz
EMD-64920,9VBD,2026-04-08,2026-04-08,3.26,gt3.0_le4.0,521,1,0.0,0.7685,9VBD,15,0.000,2,893,1,128.0,https://files.rcsb.org/download/9VBD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64920/map/emd_64920.map.gz
EMD-54346,9RWS,2026-04-29,2026-04-29,3.28,gt3.0_le4.0,512,1,0.0,0.8467,9RWS,16,0.441,1,551,1,128.0,https://files.rcsb.org/download/9RWS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54346/map/emd_54346.map.gz
EMD-66226,9WTX,2026-09-23,2026-09-23,3.37,gt3.0_le4.0,500,1,0.0,0.7238,9US8,17,0.763,3,551,1,128.0,https://files.rcsb.org/download/9WTX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66226/map/emd_66226.map.gz
EMD-73762,9Z24,2026-06-10,2026-06-10,3.1,gt3.0_le4.0,338,1,0.0,0.8749,9Z1X,18,1.000,4,1214,1,221.2,https://files.rcsb.org/download/9Z24.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73762/map/emd_73762.map.gz
EMD-80558,26DK,2026-08-26,2026-08-26,3.5,gt3.0_le4.0,191,1,0.0,0.7407,26DK,19,0.000,2,197,1,16.0,https://files.rcsb.org/download/26DK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/map/emd_80558.map.gz


In [ ]:
EXPECTED = {'scripts/nbtools.py': '9bda95fce7f7fbe0c373e315740f555fc27942a08971ce2fe1b5cae6af6d748d', 'scripts/map_survey.py': 'bf4bf51089cfaaf19920ebfd6f2c1a383b8e047b8abf6223cad93f275487ae59', 'data/entries_2026.csv': 'cb695bb04ef09bf48f8c2848f91a2580c0b3c15f2a8f1e580659d86f0097a5da'}
bad = [p for p, h in EXPECTED.items() if not os.path.exists(p) or sha256_file(p) != h]
if bad:
    fail("these files were not written as expected: %s. Use Run all again; if it happens again, send me this message." % ", ".join(bad))
else:
    log("all %d files written and checked (hashes match)" % len(EXPECTED))


## Step 2b. Progress lines, a zip that is refreshed, and the last cell that works after a restart

This cell adds three things that the first pilot lacked: a progress line every few minutes while a program is silent, the file `state.json`, and a zip
(`survey_partial.zip` in the output files) that is written again after every stage and every ten minutes, so a stop in the middle never leaves nothing.

In [ ]:
sys.path.insert(0, os.path.join(WORK, "scripts"))
import nbtools

# Kaggle's kernel sets MPLBACKEND=module://matplotlib_inline.backend_inline. Every program started from this notebook inherits it, and `import matplotlib.pyplot`
# then fails in any environment without matplotlib-inline (ModelAngelo's, in the first pilot: the run stopped after 5 minutes). Agg needs nothing.
os.environ["MPLBACKEND"] = "Agg"

NOTEBOOK = "survey"
STATE_FILE = os.path.join(WORK, "state.json")
PARTIAL_ZIP = nbtools.partial_zip_path(OUT_DIR, NOTEBOOK)
PACK_LOCK = threading.Lock()
LAST_PARTIAL = {"t": 0.0}


def save_state():
    try:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(STATE, f, indent=1, default=str)
    except Exception as ex:
        log("could not save state.json: %s" % ex)


def pack_now(min_gap=0):
    """Writes the partial zip again (same name every time). min_gap: do nothing if the last one is younger than this many seconds."""
    with PACK_LOCK:
        if time.time() - LAST_PARTIAL["t"] < min_gap:
            return
        LAST_PARTIAL["t"] = time.time()
        try:
            save_state()
            nbtools.pack(WORK, PARTIAL_ZIP, NOTEBOOK + "_partial", secrets=SECRETS)
        except Exception as ex:
            log("the partial zip could not be written: %s" % ex)


BEAT = float(os.environ.get("MICA_BEAT", "120"))          # seconds between progress lines (the tests make it short)


def run_watched(cmd, label, watch=(), beat=None, zip_every=600, **kw):
    """run() with a progress line every `beat` seconds (time, load, free RAM, busiest processes, last line of the files in `watch`)."""
    hb = nbtools.Heartbeat(label, log, watch=watch, every=beat or BEAT, on_tick=lambda: pack_now(zip_every))
    hb.start()
    try:
        return run(cmd, **kw)
    finally:
        hb.stop()


def finish(complete, extra_info=None):
    """Writes run_info.json and the final zip; returns the path of the zip."""
    info = {"notebook": NOTEBOOK, "started_utc": STATE["started"], "finished_utc": nbtools.utc_now().isoformat(), "elapsed_minutes": round((time.time() - STATE["t0"]) / 60),
            "problem": STATE["problem"], "complete": complete, "platform": PLATFORM, "python": platform.python_version(), "os": platform.platform(), "gpu": STATE["gpu"],
            "cpu_cores": os.cpu_count(), "ram_gb": meminfo_gb(), "state": STATE}
    info.update(extra_info or {})
    with open(os.path.join(WORK, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    save_state()
    path, n, left_out = nbtools.final_pack(WORK, OUT_DIR, NOTEBOOK, complete, secrets=SECRETS)
    log("zip written: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        log("left out of the zip: %s" % "; ".join(left_out))
    return path


def cpu_facts():
    """How many processors the program sees and how many it may really use (a container can show more cores than its limit allows)."""
    import multiprocessing
    facts = {"os.cpu_count": os.cpu_count(), "multiprocessing.cpu_count": multiprocessing.cpu_count(),
             "usable (sched_getaffinity)": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None, "nproc": sh("nproc 2>&1"),
             "cgroup v2 cpu.max": sh("cat /sys/fs/cgroup/cpu.max 2>&1"), "cgroup v1 quota": sh("cat /sys/fs/cgroup/cpu/cpu.cfs_quota_us 2>&1"),
             "MICA's nproc formula min(max(int(0.6 * cpu_count), 1), 48)": min(max(int(0.6 * multiprocessing.cpu_count()), 1), 48)}
    with open(os.path.join(WORK, "cpu_facts.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join("%s: %s" % kv for kv in facts.items()) + "\n")
    log("processors: " + " | ".join("%s %s" % (k.split(" ")[0], v) for k, v in facts.items()))
    STATE["cpu_facts"] = facts


save_state()
cpu_facts()
log("progress lines, state.json and the partial zip (%s) are ready" % PARTIAL_ZIP)


## Step 3. The survey

For each entry: the map and the deposited model are downloaded (the map is deleted again after the entry), the numbers are measured by `scripts/map_survey.py`, and
one line is added to `survey.csv` and the zip is refreshed. An entry that fails (download, unreadable file) is recorded with the reason and the next one follows.

In [ ]:
import map_survey

RCSB_API = "https://data.rcsb.org/rest/v1/core"
ENTRIES = []                                   # [] = every entry of the table
MAPS_DIR = os.path.join(BIG, "maps")
COMPONENT_COLUMNS = ["entry", "rank", "volume_A3", "voxels_in_model_region", "centre_x", "centre_y", "centre_z"]
RESULTS = []
def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path, columns, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns, lineterminator="\n", extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def download(url, dest, min_bytes=1, tries=3):
    import requests
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    last = ""
    for attempt in range(1, tries + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 120)) as r:
                r.raise_for_status()
                done = 0
                with open(dest + ".part", "wb") as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
                        done += len(block)
            if done < min_bytes:
                raise IOError("only %d bytes arrived" % done)
            os.replace(dest + ".part", dest)
            return dest
        except (requests.RequestException, IOError) as ex:
            last = str(ex)[:200]
            log("download attempt %d of %d failed (%s): %s" % (attempt, tries, url, last))
            time.sleep(5 * attempt)
    raise RuntimeError("could not download %s: %s" % (url, last))


def contour_from_header_xml(text):
    """(contour level, resolution) from an EMDB header file: the primary contour and the final reconstruction resolution. None where missing."""
    import xml.etree.ElementTree as ET
    root = ET.fromstring(text)
    local = lambda el: el.tag.rsplit("}", 1)[-1]
    level, resolution = None, None
    contours = [el for el in root.iter() if local(el) == "contour"]
    contours.sort(key=lambda el: el.get("primary", "false").lower() != "true")
    for el in contours:
        for ch in el:
            if local(ch) == "level" and ch.text and level is None:
                level = float(ch.text)
    for el in root.iter():
        if local(el) == "final_reconstruction":
            for ch in el:
                if local(ch) == "resolution" and ch.text and resolution is None:
                    resolution = float(ch.text)
    return level, resolution


def header_facts(text):
    """Symmetry and copy numbers from an EMDB header file (the tags are found by name, so the version of the schema does not matter)."""
    import xml.etree.ElementTree as ET
    root = ET.fromstring(text)
    local = lambda el: el.tag.rsplit("}", 1)[-1]
    texts = lambda name: "; ".join(dict.fromkeys(el.text.strip() for el in root.iter() if local(el) == name and el.text and el.text.strip()))
    sample = [el for el in root.iter() if local(el) == "sample"]
    names = [ch.text.strip() for el in sample[:1] for ch in el if local(ch) == "name" and ch.text]
    return {"point_group": texts("point_group"), "number_of_copies_header": texts("number_of_copies"), "oligomeric_state_header": texts("oligomeric_state"),
            "sample_name": (names[0] if names else "")[:120]}


def rcsb_facts(pdb):
    """What RCSB says about the assembly. Every field is optional: it only goes into the table."""
    import requests
    out = {}
    try:
        e = requests.get("%s/entry/%s" % (RCSB_API, pdb), timeout=60).json()
        out["assembly_count"] = e.get("rcsb_entry_info", {}).get("assembly_count", "")
    except Exception as ex:
        out["rcsb_note"] = "RCSB entry: %s" % str(ex)[:80]
    try:
        a = requests.get("%s/assembly/%s/1" % (RCSB_API, pdb), timeout=60).json()
        out["assembly1_polymer_instances"] = a.get("rcsb_assembly_info", {}).get("polymer_instance_count", "")
        out["assembly1_oligomeric_count"] = a.get("pdbx_struct_assembly", {}).get("oligomeric_count", "")
        out["symmetry_rcsb"] = "; ".join(("%s %s" % (s.get("symbol", ""), s.get("kind", ""))).strip() for s in (a.get("rcsb_struct_symmetry") or []))
    except Exception as ex:
        out["rcsb_note"] = (out.get("rcsb_note", "") + " RCSB assembly: %s" % str(ex)[:80]).strip()
    return out


def survey_entry(row):
    """(row of survey.csv, components) for one entry of the table."""
    entry, number = row["pdb_id"], row["emdb_id"].split("-")[-1]
    out = dict.fromkeys(map_survey.SURVEY_COLUMNS, "")
    out.update(entry=entry, emdb_id=row["emdb_id"], resolution_table_A=row["resolution_A"])
    t0 = time.time()
    gz = download(row["map_url"], os.path.join(BIG, "downloads", os.path.basename(row["map_url"])), 100000)
    map_path = map_survey.gunzip_if_needed(gz, os.path.join(MAPS_DIR, "emd_%s.map" % number))
    out["map_file_mb"] = round(os.path.getsize(map_path) / 1e6, 1)
    out["map_sha256"] = sha256_file(map_path)
    cif = download(row["cif_url"], os.path.join(WORK, "native", entry + ".cif"), 1000)
    notes, contour, resolution = [], None, None
    try:
        xml_url = re.sub(r"/map/[^/]*$", "/header/emd-%s.xml" % number, row["map_url"])
        with open(download(xml_url, os.path.join(BIG, "downloads", "emd-%s.xml" % number), 200), encoding="utf-8") as f:
            text = f.read()
        contour, resolution = contour_from_header_xml(text)
        out.update(header_facts(text))
    except Exception as ex:
        notes.append("EMDB header file: %s" % str(ex)[:120])
    out["resolution_header_A"] = resolution if resolution is not None else ""
    facts = rcsb_facts(entry)
    if facts.get("rcsb_note"):
        notes.append(facts.pop("rcsb_note"))
    out.update(facts)
    res = map_survey.survey(map_path, cif, contour, resolution if resolution is not None else float(row["resolution_A"]))
    comps = res.pop("components")
    out.update(res)
    out["note"] = "; ".join(x for x in [out.get("note", "")] + notes if x)
    out["seconds"] = round(time.time() - t0, 1)                      # downloads included
    return out, comps


def clean_up():
    for pattern in (os.path.join(MAPS_DIR, "*.map"), os.path.join(BIG, "downloads", "*.map.gz")):
        for p in glob.glob(pattern):
            try:
                os.remove(p)
            except OSError:
                pass


def summary_table(results):
    log("\n%-5s %-13s %6s %7s %5s %8s %7s %7s %6s %8s %8s  %s" % ("entry", "grid", "voxel", "box A", "res", "contour", "copies", "inside", "pieces", "Mrefl", "dock x", "symmetry / note"))
    for o in results:
        if not o.get("grid_x"):
            log("%-5s %s" % (o["entry"], o.get("note", "")))
            continue
        log("%-5s %-13s %6s %7s %5s %8.3g %7s %7s %6s %8s %8s  %s%s" % (
            o["entry"], "%sx%sx%s" % (o["grid_x"], o["grid_y"], o["grid_z"]), o["voxel_A"], o["box_x_A"], o["resolution_header_A"] or o["resolution_table_A"], o["contour_level"],
            o["copies_of_model_estimate"], o["above_fraction_in_model"], o["components_ge_5000A3"], o["reflections_full_M"], o["docking_load_vs_26DK"],
            o.get("point_group") or o.get("symmetry_rcsb") or "-", ("  | " + o["note"]) if o.get("note") else ""))
    log("copies = volume above the contour level / the part of it within 3 A of the deposited atoms; inside = that part as a fraction; pieces = connected pieces of at least 5000 A^3;")
    log("Mrefl = millions of reflections of the whole box at the map resolution (Phaser's work in the docking); dock x = Mrefl relative to 26DK, which did not finish in 20 minutes.")


def run_survey():
    rows = {r["pdb_id"]: r for r in read_csv("data/entries_2026.csv")}
    comps_all = []
    for entry in (ENTRIES or list(rows)):
        res = STATE["entries"][entry] = {"entry": entry, "status": "failed", "note": ""}
        log("\n" + "=" * 72 + "\nENTRY %s\n" % entry + "=" * 72)
        try:
            out, comps = survey_entry(rows[entry])
            RESULTS.append(out)
            comps_all += [dict(c, entry=entry) for c in comps]
            res.update(status="ok", note=out["note"])
            log("%s: grid %sx%sx%s, voxel %s A, box %s A, contour %s, about %s copies of the model, %s pieces, %s M reflections (%s s)" % (
                entry, out["grid_x"], out["grid_y"], out["grid_z"], out["voxel_A"], out["box_x_A"], out["contour_level"], out["copies_of_model_estimate"],
                out["components_ge_5000A3"], out["reflections_full_M"], out["seconds"]))
            if out["note"]:
                log("    note: %s" % out["note"])
        except Exception:
            import traceback
            tb = traceback.format_exc()
            res["note"] = tb.strip().splitlines()[-1][:300]
            log("ENTRY %s FAILED:\n%s" % (entry, tb))
            RESULTS.append({"entry": entry, "emdb_id": rows.get(entry, {}).get("emdb_id", ""), "resolution_table_A": rows.get(entry, {}).get("resolution_A", ""),
                            "note": "FAILED: " + res["note"]})
        finally:
            clean_up()
        write_csv("survey.csv", map_survey.SURVEY_COLUMNS, RESULTS)
        write_csv("survey_components.csv", COMPONENT_COLUMNS, comps_all)
        pack_now()
    summary_table(RESULTS)
    STATE["complete"] = bool(STATE["entries"]) and all(e["status"] == "ok" for e in STATE["entries"].values())
    if not STATE["complete"]:
        STATE["stopped_at"] = "failed entries: " + ", ".join(e["entry"] for e in STATE["entries"].values() if e["status"] != "ok")


if STATE["ok"]:
    try:
        run_survey()
    except Exception:
        import traceback
        STATE["stopped_at"] = "survey"
        log("the survey stopped by an error:\n%s" % traceback.format_exc())
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Last step. Pack the results

Run this cell last, also after a problem. It works after a restart of the kernel too: it then packs the files that are on disk.

In [ ]:
import os, sys
try:
    finish
except NameError:                                  # fresh kernel: nothing of the earlier cells is left, only the files on disk
    work = None
    for cand in (os.environ.get("MICA_WORK"), "/kaggle/working/pilot", "/content/pilot", os.path.abspath("pilot")):
        if cand and os.path.isfile(os.path.join(cand, "scripts", "nbtools.py")):
            work = cand
            break
    if work is None:
        print("RECOVERY: the work folder was not found, so there is nothing to pack. Run the notebook from the top.")
    else:
        sys.path.insert(0, os.path.join(work, "scripts"))
        import nbtools
        nbtools.recover("survey")
else:
    zip_path = finish(bool(STATE["ok"] and STATE.get("complete")), {"input_sha256": globals().get("EXPECTED", {}), "entries": sorted(STATE["entries"]), "map_survey": "scripts/map_survey.py"})
    if STATE["ok"] and STATE.get("complete"):
        log("\nRESULT: OK. Download %s (Kaggle: the output files of this version) and send it to me." % zip_path)
    elif STATE["ok"]:
        log("\nRESULT: NOT COMPLETE (%s). Send me %s; it shows where it stopped." % (STATE.get("stopped_at") or "a step did not finish", zip_path))
    else:
        log("\nRESULT: STOPPED (%s). Send me %s." % (STATE["problem"], zip_path))
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        log("(Kaggle: open the version, then Output, then download the zip. Colab: the file list on the left.)")
